# BUS-BRA Verification

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import skimage.io as io

RAW_DATA_DIR = Path("/scratch/data/busbra_raw_20260518/BUSBRA/")

In [ ]:
# Get a list of the file extensions of the images in `Images` and the masks in
# `Masks`
image_extensions = set()
mask_extensions = set()

for image_path in (RAW_DATA_DIR / "Images").glob("*.*"):
    image_extensions.add(image_path.suffix)

for mask_path in (RAW_DATA_DIR / "Masks").glob("*.*"):
    mask_extensions.add(mask_path.suffix)

print("Image extensions:", image_extensions)
print("Mask extensions:", mask_extensions)

In [ ]:
# The images are named "bus_{patient:04d}-{l|r}.png". Load the images using
# scikit-image and report:
#   - number of patients
#   - number of images
#   - mean number of images per patient
#   - number of images with N channels
#   - min/max pixel values
#   - min/mean/std/max height and width

image_paths = list((RAW_DATA_DIR / "Images").glob("*.*"))
patient_image_counts = {}
channel_counts = {}
min_pixel_value = float("inf")
max_pixel_value = float("-inf")
heights = []
widths = []
for image_path in image_paths:
    # Extract patient ID from the filename
    filename = image_path.stem  # Get the filename without extension
    patient_id = filename.split("-")[0]  # Get the part before the hyphen

    # Load the image
    image = io.imread(image_path)

    # Update patient image counts
    if patient_id not in patient_image_counts:
        patient_image_counts[patient_id] = 0
    patient_image_counts[patient_id] += 1

    # Update channel counts
    channels = image.shape[2] if len(image.shape) == 3 else 1
    if channels not in channel_counts:
        channel_counts[channels] = 0
    channel_counts[channels] += 1

    # Update pixel values and dimensions
    min_pixel_value = min(min_pixel_value, image.min())
    max_pixel_value = max(max_pixel_value, image.max())
    heights.append(image.shape[0])
    widths.append(image.shape[1])

num_patients = len(patient_image_counts)
num_images = len(image_paths)
mean_images_per_patient = num_images / num_patients
min_height = min(heights)
max_height = max(heights)
mean_height = np.mean(heights)
std_height = np.std(heights)
min_width = min(widths)
max_width = max(widths)
mean_width = np.mean(widths)
std_width = np.std(widths)

print(f"Number of patients: {num_patients}")
print(f"Number of images: {num_images}")
print(f"Mean number of images per patient: {mean_images_per_patient:.2f}")
print("Number of images with N channels:", channel_counts)
print(f"Min pixel value: {min_pixel_value}")
print(f"Max pixel value: {max_pixel_value}")
print(
    f"Height: mean={mean_height:.2f}, std={std_height:.2f} (min={min_height}, max={max_height})"  # noqa: E501
)
print(
    f"Width: mean={mean_width:.2f}, std={std_width:.2f} (min={min_width}, max={max_width})"  # noqa: E501
)

In [ ]:
# The masks are named "mask_{patient:04d}-{l|r}.png". Check that there is one
# mask per image and load the masks using scikit-image and report:
#   - number of masks with N channels
#   - min/max pixel values

mask_paths = list((RAW_DATA_DIR / "Masks").glob("*.*"))
mask_channel_counts = {}
mask_min_pixel_value = float("inf")
mask_max_pixel_value = float("-inf")

images = set(image_path.stem.removeprefix("bus_") for image_path in image_paths)
masks = set(mask_path.stem.removeprefix("mask_") for mask_path in mask_paths)
assert images == masks, "Mismatch between images and masks"

for mask_path in mask_paths:
    mask = io.imread(mask_path)

    # Update channel counts
    channels = mask.shape[2] if len(mask.shape) == 3 else 1
    if channels not in mask_channel_counts:
        mask_channel_counts[channels] = 0
    mask_channel_counts[channels] += 1

    # Update pixel values
    mask_min_pixel_value = min(mask_min_pixel_value, mask.min())
    mask_max_pixel_value = max(mask_max_pixel_value, mask.max())

print("Number of masks with N channels:", mask_channel_counts)
print(f"Min pixel value in masks: {mask_min_pixel_value}")
print(f"Max pixel value in masks: {mask_max_pixel_value}")

In [ ]:
# Display an image and its corresponding mask

image_path = image_paths[0]
mask_path = RAW_DATA_DIR / "Masks" / image_path.name.replace("bus_", "mask_")
image = io.imread(image_path)
mask = io.imread(mask_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image)
axes[0].set_title("Image")
axes[0].axis("off")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Mask")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Read the metadata `bus_data.csv`
df = pd.read_csv(RAW_DATA_DIR / "bus_data.csv")
df.head()

In [ ]:
# Plot the distributions of `Histology`, `Pathology`, `BIRADS`, and `Device` in
# a 2 x 2 grid

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sns.countplot(data=df, x="Histology", ax=axes[0, 0])
axes[0, 0].tick_params(axis="x", rotation=90)
axes[0, 0].set_title("Distribution of Histology")
sns.countplot(data=df, x="Pathology", ax=axes[0, 1])
axes[0, 1].set_title("Distribution of Pathology")
sns.countplot(data=df, x="BIRADS", ax=axes[1, 0])
axes[1, 0].set_title("Distribution of BIRADS")
sns.countplot(data=df, x="Device", ax=axes[1, 1])
axes[1, 1].set_title("Distribution of Device")
axes[1, 1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df["Histology"].value_counts()